# 02 - BankNote Authentication Modelleri

Bu notebook'ta BankNote Authentication veri seti üzerinde **ikili sınıflandırma (binary classification)** yapılmaktadır.

## İçerik
1. Veri hazırlama
2. **Scratch MLP modelleri** (5 farklı konfigürasyon)
3. Overfitting / Underfitting analizi
4. En iyi model seçimi
5. **Scikit-learn MLPClassifier**
6. **PyTorch MLP**
7. Tüm modellerin karşılaştırması

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import DataLoader
from src.mlp_scratch import MLPScratch
from src.metrics import (
    print_metrics, plot_confusion_matrix, plot_cost_curves,
    plot_accuracy_curves, compare_models
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

print('Kütüphaneler yüklendi ✅')

## 1. Veri Hazırlama

In [ ]:
loader = DataLoader(random_state=42)
loader.load_banknote('../data/BankNote_Authentication.csv')

X_train, X_val, X_test, y_train, y_val, y_test = loader.split_data(
    test_size=0.15, val_size=0.15, scaling='standard'
)

n_features = X_train.shape[1]
print(f'\nÖzellik sayısı: {n_features}')

---

## 2. Scratch MLP Modelleri

Lab dersinde oluşturulan 1 gizli katmanlı modelin devamı olarak 5 farklı konfigürasyon eğitilecektir.

| # | Model | Mimari | LR | Steps | Reg |
|---|-------|--------|----|-------|-----|
| 1 | Baseline | [4,5,1] | 0.01 | 1000 | — |
| 2 | Wider | [4,10,1] | 0.01 | 1000 | — |
| 3 | Deeper | [4,8,4,1] | 0.01 | 1000 | — |
| 4 | Regularized | [4,8,4,1] | 0.01 | 1000 | L2 λ=0.01 |
| 5 | Tuned | [4,16,8,1] | 0.05 | 2000 | L2 λ=0.001 |

### Model 1: Baseline (1 Gizli Katman, 5 Nöron)

Lab dersinde kullanılan temel model. Mimari: **[4 → 5 → 1]**

In [ ]:
model1 = MLPScratch(
    layer_dims=[n_features, 5, 1],
    learning_rate=0.01,
    n_steps=1000,
    random_state=42,
    lambda_reg=0.0,
    task='binary',
    activation_hidden='tanh'
)
print(model1)
model1.fit(X_train, y_train, X_val, y_val, print_every=200)

In [ ]:
# Maliyet Eğrisi
plot_cost_curves(model1.cost_history_train, model1.cost_history_val,
                 'Model 1: Baseline [4,5,1]', '../results/banknote/bn_model1_cost.png')

# Doğruluk Eğrisi
plot_accuracy_curves(model1.accuracy_history_train, model1.accuracy_history_val,
                     'Model 1: Baseline [4,5,1]', '../results/banknote/bn_model1_acc.png')

In [ ]:
# Test Metrikleri
y_pred_1 = model1.predict(X_test)
res1 = print_metrics(y_test, y_pred_1, 'Model 1: Baseline [4,5,1]', task='binary')
plot_confusion_matrix(y_test, y_pred_1, 'Model 1: Baseline [4,5,1]',
                      class_names=['Gerçek (0)', 'Sahte (1)'],
                      save_path='../results/banknote/bn_model1_cm.png')

### Model 2: Wider (1 Gizli Katman, 10 Nöron)

Gizli katmandaki nöron sayısı artırıldı. Mimari: **[4 → 10 → 1]**

In [ ]:
model2 = MLPScratch(
    layer_dims=[n_features, 10, 1],
    learning_rate=0.01,
    n_steps=1000,
    random_state=42,
    lambda_reg=0.0,
    task='binary',
    activation_hidden='tanh'
)
print(model2)
model2.fit(X_train, y_train, X_val, y_val, print_every=200)

In [ ]:
plot_cost_curves(model2.cost_history_train, model2.cost_history_val,
                 'Model 2: Wider [4,10,1]', '../results/banknote/bn_model2_cost.png')
plot_accuracy_curves(model2.accuracy_history_train, model2.accuracy_history_val,
                     'Model 2: Wider [4,10,1]', '../results/banknote/bn_model2_acc.png')

In [ ]:
y_pred_2 = model2.predict(X_test)
res2 = print_metrics(y_test, y_pred_2, 'Model 2: Wider [4,10,1]', task='binary')
plot_confusion_matrix(y_test, y_pred_2, 'Model 2: Wider [4,10,1]',
                      class_names=['Gerçek (0)', 'Sahte (1)'],
                      save_path='../results/banknote/bn_model2_cm.png')

### Model 3: Deeper (2 Gizli Katman)

Gizli katman sayısı 1 artırıldı. Mimari: **[4 → 8 → 4 → 1]**

In [ ]:
model3 = MLPScratch(
    layer_dims=[n_features, 8, 4, 1],
    learning_rate=0.01,
    n_steps=1000,
    random_state=42,
    lambda_reg=0.0,
    task='binary',
    activation_hidden='tanh'
)
print(model3)
model3.fit(X_train, y_train, X_val, y_val, print_every=200)

In [ ]:
plot_cost_curves(model3.cost_history_train, model3.cost_history_val,
                 'Model 3: Deeper [4,8,4,1]', '../results/banknote/bn_model3_cost.png')
plot_accuracy_curves(model3.accuracy_history_train, model3.accuracy_history_val,
                     'Model 3: Deeper [4,8,4,1]', '../results/banknote/bn_model3_acc.png')

In [ ]:
y_pred_3 = model3.predict(X_test)
res3 = print_metrics(y_test, y_pred_3, 'Model 3: Deeper [4,8,4,1]', task='binary')
plot_confusion_matrix(y_test, y_pred_3, 'Model 3: Deeper [4,8,4,1]',
                      class_names=['Gerçek (0)', 'Sahte (1)'],
                      save_path='../results/banknote/bn_model3_cm.png')

### Model 4: Regularized (2 Gizli Katman + L2)

Aynı derin mimariyle L2 regülarizasyon eklendi. Mimari: **[4 → 8 → 4 → 1]**, λ=0.01

In [ ]:
model4 = MLPScratch(
    layer_dims=[n_features, 8, 4, 1],
    learning_rate=0.01,
    n_steps=1000,
    random_state=42,
    lambda_reg=0.01,
    task='binary',
    activation_hidden='tanh'
)
print(model4)
model4.fit(X_train, y_train, X_val, y_val, print_every=200)

In [ ]:
plot_cost_curves(model4.cost_history_train, model4.cost_history_val,
                 'Model 4: Regularized [4,8,4,1] λ=0.01', '../results/banknote/bn_model4_cost.png')
plot_accuracy_curves(model4.accuracy_history_train, model4.accuracy_history_val,
                     'Model 4: Regularized [4,8,4,1] λ=0.01', '../results/banknote/bn_model4_acc.png')

In [ ]:
y_pred_4 = model4.predict(X_test)
res4 = print_metrics(y_test, y_pred_4, 'Model 4: Regularized [4,8,4,1] λ=0.01', task='binary')
plot_confusion_matrix(y_test, y_pred_4, 'Model 4: Regularized [4,8,4,1] λ=0.01',
                      class_names=['Gerçek (0)', 'Sahte (1)'],
                      save_path='../results/banknote/bn_model4_cm.png')

### Model 5: Tuned (Optimize Edilmiş)

Daha yüksek öğrenme oranı, daha fazla epoch, hafif regülarizasyon. Mimari: **[4 → 16 → 8 → 1]**, lr=0.05, λ=0.001

In [ ]:
model5 = MLPScratch(
    layer_dims=[n_features, 16, 8, 1],
    learning_rate=0.05,
    n_steps=2000,
    random_state=42,
    lambda_reg=0.001,
    task='binary',
    activation_hidden='tanh'
)
print(model5)
model5.fit(X_train, y_train, X_val, y_val, print_every=400)

In [ ]:
plot_cost_curves(model5.cost_history_train, model5.cost_history_val,
                 'Model 5: Tuned [4,16,8,1]', '../results/banknote/bn_model5_cost.png')
plot_accuracy_curves(model5.accuracy_history_train, model5.accuracy_history_val,
                     'Model 5: Tuned [4,16,8,1]', '../results/banknote/bn_model5_acc.png')

In [ ]:
y_pred_5 = model5.predict(X_test)
res5 = print_metrics(y_test, y_pred_5, 'Model 5: Tuned [4,16,8,1]', task='binary')
plot_confusion_matrix(y_test, y_pred_5, 'Model 5: Tuned [4,16,8,1]',
                      class_names=['Gerçek (0)', 'Sahte (1)'],
                      save_path='../results/banknote/bn_model5_cm.png')

---

## 3. Overfitting / Underfitting Analizi

Tüm modellerin train ve validation maliyet eğrileri karşılaştırılarak **yüksek varyans (overfitting)** veya **yüksek bias (underfitting)** durumları incelenir.

In [ ]:
os.makedirs('../results/banknote', exist_ok=True)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('BankNote - Overfitting/Underfitting Analizi', fontsize=16, fontweight='bold')

models_info = [
    (model1, 'Model 1: Baseline [4,5,1]'),
    (model2, 'Model 2: Wider [4,10,1]'),
    (model3, 'Model 3: Deeper [4,8,4,1]'),
    (model4, 'Model 4: Reg [4,8,4,1] λ=0.01'),
    (model5, 'Model 5: Tuned [4,16,8,1]')
]

for idx, (model, name) in enumerate(models_info):
    ax = axes[idx // 3, idx % 3]
    ax.plot(model.cost_history_train, label='Train', color='#2196F3', linewidth=1.5)
    ax.plot(model.cost_history_val, label='Validation', color='#FF5722', linewidth=1.5, linestyle='--')
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Cost')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Boş subplot'u gizle
axes[1, 2].set_visible(False)

plt.tight_layout()
plt.savefig('../results/banknote/bn_overfitting_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Overfitting/Underfitting Yorumu:')
for model, name in models_info:
    train_final = model.cost_history_train[-1]
    val_final = model.cost_history_val[-1]
    gap = abs(val_final - train_final)
    
    if train_final > 0.3:
        status = '⚠️ Yüksek Bias (Underfitting)'
    elif gap > 0.1:
        status = '⚠️ Yüksek Varyans (Overfitting)'
    else:
        status = '✅ İyi Uyum (Good Fit)'
    
    print(f'  {name}: Train={train_final:.4f}, Val={val_final:.4f}, Gap={gap:.4f} → {status}')

---

## 4. En İyi Model Seçimi

Accuracy ≥ %90 olan modeller arasından **en düşük n_steps** değerine sahip model seçilir.

In [ ]:
scratch_results = {
    'Model 1: Baseline [4,5,1]': {**res1, 'n_steps': 1000},
    'Model 2: Wider [4,10,1]': {**res2, 'n_steps': 1000},
    'Model 3: Deeper [4,8,4,1]': {**res3, 'n_steps': 1000},
    'Model 4: Reg [4,8,4,1]': {**res4, 'n_steps': 1000},
    'Model 5: Tuned [4,16,8,1]': {**res5, 'n_steps': 2000}
}

print('\n🏆 Model Seçimi (Accuracy ≥ 90%):')
print('-' * 60)

qualified = {}
for name, r in scratch_results.items():
    if r['accuracy'] >= 0.90:
        qualified[name] = r
        print(f'  ✅ {name}: Acc={r["accuracy"]:.4f}, Steps={r["n_steps"]}')
    else:
        print(f'  ❌ {name}: Acc={r["accuracy"]:.4f} (≤ 90%)')

if qualified:
    best_name = min(qualified, key=lambda k: qualified[k]['n_steps'])
    print(f'\n  🥇 Seçilen Model: {best_name} (En düşük n_steps ile ≥90% accuracy)')
else:
    best_name = max(scratch_results, key=lambda k: scratch_results[k]['accuracy'])
    print(f'\n  🥇 Seçilen Model: {best_name} (En yüksek accuracy)')

---

## 5. Scikit-learn MLPClassifier

Aynı mimari, aynı eğitim/test seti, SGD optimizer ve aynı başlangıç ayarları ile sklearn MLPClassifier.

In [ ]:
from sklearn.neural_network import MLPClassifier

# Model 1 mimarisi: [4,5,1] → hidden_layer_sizes=(5,)
sklearn_model1 = MLPClassifier(
    hidden_layer_sizes=(5,),
    activation='tanh',
    solver='sgd',
    learning_rate_init=0.01,
    max_iter=1000,
    random_state=42,
    verbose=False
)
sklearn_model1.fit(X_train, y_train.ravel())
y_pred_sk1 = sklearn_model1.predict(X_test)
res_sk1 = print_metrics(y_test, y_pred_sk1, 'Sklearn: [4,5,1]', task='binary')
plot_confusion_matrix(y_test, y_pred_sk1, 'Sklearn: [4,5,1]',
                      class_names=['Gerçek (0)', 'Sahte (1)'],
                      save_path='../results/banknote/bn_sklearn1_cm.png')

In [ ]:
# Model 3 mimarisi: [4,8,4,1] → hidden_layer_sizes=(8,4)
sklearn_model2 = MLPClassifier(
    hidden_layer_sizes=(8, 4),
    activation='tanh',
    solver='sgd',
    learning_rate_init=0.01,
    max_iter=1000,
    random_state=42,
    verbose=False
)
sklearn_model2.fit(X_train, y_train.ravel())
y_pred_sk2 = sklearn_model2.predict(X_test)
res_sk2 = print_metrics(y_test, y_pred_sk2, 'Sklearn: [4,8,4,1]', task='binary')
plot_confusion_matrix(y_test, y_pred_sk2, 'Sklearn: [4,8,4,1]',
                      class_names=['Gerçek (0)', 'Sahte (1)'],
                      save_path='../results/banknote/bn_sklearn2_cm.png')

---

## 6. PyTorch MLP

Aynı mimari ve hiperparametrelerle PyTorch implementasyonu.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Tekrarlanabilirlik
torch.manual_seed(42)
np.random.seed(42)

class MLPPyTorch(nn.Module):
    """PyTorch MLP modeli."""
    def __init__(self, layer_dims, activation='tanh'):
        super(MLPPyTorch, self).__init__()
        layers = []
        for i in range(len(layer_dims) - 1):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i+1]))
            if i < len(layer_dims) - 2:  # Gizli katmanlar
                if activation == 'tanh':
                    layers.append(nn.Tanh())
                elif activation == 'relu':
                    layers.append(nn.ReLU())
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

print('PyTorch hazır ✅')

In [ ]:
def train_pytorch_model(model, X_train, y_train, X_test, y_test,
                        lr=0.01, n_epochs=1000, model_name='PyTorch Model'):
    """PyTorch modelini eğitir ve değerlendirir."""
    # NumPy -> Tensor
    X_tr = torch.FloatTensor(X_train)
    y_tr = torch.FloatTensor(y_train.reshape(-1, 1))
    X_te = torch.FloatTensor(X_test)
    y_te = torch.FloatTensor(y_test.reshape(-1, 1))
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    
    cost_history = []
    
    for epoch in range(n_epochs):
        # Forward
        outputs = model(X_tr)
        loss = criterion(outputs, y_tr)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        cost_history.append(loss.item())
        
        if epoch % 200 == 0:
            print(f'  Epoch {epoch:5d} | Loss: {loss.item():.6f}')
    
    # Değerlendirme
    model.eval()
    with torch.no_grad():
        preds = torch.sigmoid(model(X_te))
        y_pred = (preds > 0.5).float().numpy().flatten()
    
    return y_pred, cost_history

In [ ]:
# PyTorch Model 1: [4,5,1]
torch.manual_seed(42)
pt_model1 = MLPPyTorch([n_features, 5, 1], activation='tanh')
print(f'Mimari: {pt_model1}')
y_pred_pt1, pt_costs1 = train_pytorch_model(pt_model1, X_train, y_train, X_test, y_test,
                                            lr=0.01, n_epochs=1000, model_name='PyTorch [4,5,1]')
res_pt1 = print_metrics(y_test, y_pred_pt1, 'PyTorch: [4,5,1]', task='binary')
plot_confusion_matrix(y_test, y_pred_pt1, 'PyTorch: [4,5,1]',
                      class_names=['Gerçek (0)', 'Sahte (1)'],
                      save_path='../results/banknote/bn_pytorch1_cm.png')

In [ ]:
# PyTorch Model 2: [4,8,4,1]
torch.manual_seed(42)
pt_model2 = MLPPyTorch([n_features, 8, 4, 1], activation='tanh')
y_pred_pt2, pt_costs2 = train_pytorch_model(pt_model2, X_train, y_train, X_test, y_test,
                                            lr=0.01, n_epochs=1000, model_name='PyTorch [4,8,4,1]')
res_pt2 = print_metrics(y_test, y_pred_pt2, 'PyTorch: [4,8,4,1]', task='binary')
plot_confusion_matrix(y_test, y_pred_pt2, 'PyTorch: [4,8,4,1]',
                      class_names=['Gerçek (0)', 'Sahte (1)'],
                      save_path='../results/banknote/bn_pytorch2_cm.png')

---

## 7. Tüm Modellerin Karşılaştırması

In [ ]:
all_results = {
    'Scratch [4,5,1]': res1,
    'Scratch [4,10,1]': res2,
    'Scratch [4,8,4,1]': res3,
    'Scratch [4,8,4,1]+L2': res4,
    'Scratch [4,16,8,1]': res5,
    'Sklearn [4,5,1]': res_sk1,
    'Sklearn [4,8,4,1]': res_sk2,
    'PyTorch [4,5,1]': res_pt1,
    'PyTorch [4,8,4,1]': res_pt2
}

comparison_df = compare_models(all_results, save_path='../results/banknote/bn_model_comparison.png')

In [ ]:
# Scratch vs Sklearn vs PyTorch (aynı mimari) karşılaştırması
print('\n📊 Aynı Mimari Karşılaştırması: [4,5,1]')
print(f"  Scratch  : Acc={res1['accuracy']:.4f}")
print(f"  Sklearn  : Acc={res_sk1['accuracy']:.4f}")
print(f"  PyTorch  : Acc={res_pt1['accuracy']:.4f}")

print('\n📊 Aynı Mimari Karşılaştırması: [4,8,4,1]')
print(f"  Scratch  : Acc={res3['accuracy']:.4f}")
print(f"  Sklearn  : Acc={res_sk2['accuracy']:.4f}")
print(f"  PyTorch  : Acc={res_pt2['accuracy']:.4f}")

---

## Sonuç

Bu notebook'ta BankNote Authentication veri seti üzerinde:

- 5 farklı scratch MLP modeli eğitildi
- Overfitting/Underfitting analizi yapıldı
- En iyi model seçildi
- Aynı mimariler sklearn ve PyTorch ile tekrar oluşturuldu
- Tüm modeller karşılaştırıldı

Bir sonraki notebook'ta (`03_WineQuality_MultiClass.ipynb`) Wine Quality veri seti ile çoklu sınıflandırma yapılacaktır.